In [1]:
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

stopword = StopWordRemoverFactory().create_stop_word_remover()

dokumen = [
    "Sistem komputer terdiri dari perangkat keras dan perangkat lunak.",
    "Jaringan syaraf tiruan sering digunakan untuk pengembangan kecerdasan buatan.",
    "Sistem temu kembali informasi berguna memproses dokumen teks.",
    "Algoritma data mining bertugas mengekstraksi informasi dari data yang besar."
]

In [2]:
docs_clean = []
for doc in dokumen:
    doc_clean = doc.lower()
    doc_clean = re.sub(r'[^a-z\s]', ' ', doc_clean)
    doc_stop = stopword.remove(doc_clean)
    doc_final = re.sub(r'\s+', ' ', doc_stop).strip()
    docs_clean.append(doc_final)

print("=== DOKUMEN SETELAH PREPROCESSING SEDERHANA ===")
for i, d in enumerate(docs_clean):
    print(f"D{i+1}: {d}")

=== DOKUMEN SETELAH PREPROCESSING SEDERHANA ===
D1: sistem komputer terdiri perangkat keras perangkat lunak
D2: jaringan syaraf tiruan sering digunakan pengembangan kecerdasan buatan
D3: sistem temu informasi berguna memproses dokumen teks
D4: algoritma data mining bertugas mengekstraksi informasi data besar


In [3]:
vocab = sorted(list(set(" ".join(docs_clean).split())))

tf_matrix = []
for doc in docs_clean:
    tokens = doc.split()
    tf_row = [tokens.count(term) for term in vocab]
    tf_matrix.append(tf_row)

df_tf = pd.DataFrame(tf_matrix, columns=vocab, index=[f"D{i+1}" for i in range(len(docs_clean))])

N = len(docs_clean)
df_counts = (df_tf > 0).sum(axis=0)
idf_manual = np.log10(N / df_counts)

df_tfidf_manual = df_tf.astype(float).copy()
for col in vocab:
    df_tfidf_manual[col] = df_tfidf_manual[col] * idf_manual[col]

print("=== MATRIKS TF-IDF (PERHITUNGAN MANUAL) ===")
display(df_tfidf_manual.round(4))

=== MATRIKS TF-IDF (PERHITUNGAN MANUAL) ===


,algoritma,berguna,bertugas,besar,buatan,data,digunakan,dokumen,informasi,jaringan,...,mining,pengembangan,perangkat,sering,sistem,syaraf,teks,temu,terdiri,tiruan
D1,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.000,0.0000,...,0.0000,0.0000,1.2041,0.0000,0.301,0.0000,0.0000,0.0000,0.6021,0.0000
D2,0.0000,0.0000,0.0000,0.0000,0.6021,0.0000,0.6021,0.0000,0.000,0.6021,...,0.0000,0.6021,0.0000,0.6021,0.000,0.6021,0.0000,0.0000,0.0000,0.6021
D3,0.0000,0.6021,0.0000,0.0000,0.0000,0.0000,0.0000,0.6021,0.301,0.0000,...,0.0000,0.0000,0.0000,0.0000,0.301,0.0000,0.6021,0.6021,0.0000,0.0000
D4,0.6021,0.0000,0.6021,0.6021,0.0000,1.2041,0.0000,0.0000,0.301,0.0000,...,0.6021,0.0000,0.0000,0.0000,0.000,0.0000,0.0000,0.0000,0.0000,0.0000


In [4]:
vectorizer = TfidfVectorizer()
tfidf_sklearn_matrix = vectorizer.fit_transform(docs_clean)

df_tfidf_sklearn = pd.DataFrame(
    tfidf_sklearn_matrix.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=[f"D{i+1}" for i in range(len(docs_clean))]
)

print("=== MATRIKS TF-IDF (SCIKIT-LEARN) ===")
display(df_tfidf_sklearn.round(4))

=== MATRIKS TF-IDF (SCIKIT-LEARN) ===


,algoritma,berguna,bertugas,besar,buatan,data,digunakan,dokumen,informasi,jaringan,...,mining,pengembangan,perangkat,sering,sistem,syaraf,teks,temu,terdiri,tiruan
D1,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0000,0.6811,0.0000,0.2685,0.0000,0.0000,0.0000,0.3406,0.0000
D2,0.0000,0.0000,0.0000,0.0000,0.3536,0.0000,0.3536,0.0000,0.0000,0.3536,...,0.0000,0.3536,0.0000,0.3536,0.0000,0.3536,0.0000,0.0000,0.0000,0.3536
D3,0.0000,0.4002,0.0000,0.0000,0.0000,0.0000,0.0000,0.4002,0.3155,0.0000,...,0.0000,0.0000,0.0000,0.0000,0.3155,0.0000,0.4002,0.4002,0.0000,0.0000
D4,0.3224,0.0000,0.3224,0.3224,0.0000,0.6448,0.0000,0.0000,0.2542,0.0000,...,0.3224,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000


## Analisis Hasil Pembobotan TF-IDF

1. **Term dengan Bobot Tertinggi pada Masing-Masing Dokumen:**
   - **D1:** Term **“keras”**, **“lunak”**, dan **“perangkat”** memegang nilai tertinggi.
   - **D2:** Term **“buatan”**, **“jaringan”**, **“kecerdasan”**, **“pengembangan”**, **“sering”**, **“syaraf”**, dan **“tiruan”** memiliki nilai pembobotan tertinggi secara merata.
   - **D3:** Term **“berguna”**, **“kembali”**, **“memproses”**, dan **“teks”** mendominasi pembobotan.
   - **D4:** Term **“algoritma”**, **“bertugas”**, **“besar”**, **“data”**, dan **“mengekstraksi”** memperoleh nilai tertinggi.

2. **Pentingnya Term-Term Tersebut:**
   Term-term ini menjadi sangat penting karena merepresentasikan topik atau identitas utama dari masing-masing dokumen. Algoritma TF-IDF secara otomatis memberikan bobot lebih tinggi pada kata yang sering muncul di dalam suatu dokumen tertentu (*Term Frequency* tinggi) namun jarang ditemukan di dokumen lainnya (*Document Frequency* rendah).

3. **Perbandingan Manual vs Scikit-Learn:**
   Terdapat perbedaan angka absolut antara perhitungan manual dan *library* scikit-learn. Perbedaan ini wajar terjadi karena scikit-learn menggunakan rumus logaritma alami ($\ln$), menambahkan *smoothing* pada IDF, serta menerapkan normalisasi vektor $L_2$ (*Euclidean norm*) pada hasil akhirnya agar panjang dokumen tidak memengaruhi skor kemiripan. Meskipun nilainya berbeda, *ranking* keterkaitan antar kata dalam dokumen tetap konsisten.